In [ ]:
import json
import pandas as pd


with open("raw_pages.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.columns)


print(df.head())

Index(['url', 'http_headers', 'links', 'nb_links', 'saved_at', 'scraped_at',
       'status_code', 'text', 'title', 'domain', 'label'],
      dtype='str')
                                                 url  \
0                             https://www.lemonde.fr   
1                                 https://www.rt.com   
2                            https://sputniknews.com   
3  https://boutique.lemonde.fr/nouveautes/4361-mo...   
4                  https://ensecurite.sourcesure.eu/   

                                        http_headers  \
0  {'Connection': 'keep-alive', 'Content-Length':...   
1  {'Server': 'ddos-guard', 'Connection': 'keep-a...   
2  {'Server': 'QRATOR', 'Date': 'Thu, 30 Apr 2026...   
3                                                NaN   
4                                                NaN   

                                               links  nb_links  \
0  [https://ateliers.lemonde.fr/armelle-rancillac...        86   
1  [https://francais.rt.com/, https://a

In [ ]:
from bs4 import BeautifulSoup

def clean_html(html_content):

    
    soup = BeautifulSoup(html_content, "lxml")

   
    text = soup.get_text(separator=" ")

    return text

In [3]:
df["text"] = df["text"].apply(clean_html)

C:\Users\nabil\AppData\Local\Temp\ipykernel_17320\1777872155.py:6: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_content, "lxml")


In [4]:
! pip install scikit-learn vaderSentiment

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

In [6]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [7]:
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):

    if not isinstance(text, str):
        return ""

    # Minuscule
    text = text.lower()

    # Supprimer URLs
    text = re.sub(r"http\S+", "", text)

    # Supprimer nombres
    text = re.sub(r"\d+", "", text)

    # Supprimer ponctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenization simple
    tokens = text.split()

    # Supprimer stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Supprimer petits mots
    tokens = [word for word in tokens if len(word) > 2]

    # Rejoindre
    return " ".join(tokens)

In [8]:
import re
import string
from nltk.tokenize import word_tokenize
df["clean_text"] = df["text"].apply(preprocess_text)

In [9]:
texts = df["clean_text"].fillna("").astype(str).tolist()

In [10]:
import os

os.makedirs("datasets", exist_ok=True)

In [11]:
df.to_csv("datasets/clean_pages.csv", index=False)

#### Feature Engineering (TF-IDF)

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Charger données
df = pd.read_csv("datasets/clean_pages.csv")

# Sécuriser la colonne texte
df["clean_text"] = df["clean_text"].fillna("").astype(str)

# Texte
texts = df["clean_text"].tolist()

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(texts)

print(X.shape)

(2200, 5000)


In [13]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:100])

['aaron' 'abbas' 'abide' 'ability' 'able' 'abonnement' 'abonnements'
 'abonner' 'abonné' 'abonnés' 'abuse' 'academy' 'accents' 'accept'
 'acceptance' 'access' 'accessed' 'accessibility' 'accessibilité'
 'accessible' 'accessing' 'accessoires' 'accessories' 'accordance'
 'according' 'account' 'accounts' 'accueil' 'accurate' 'accused' 'accèdes'
 'accès' 'accéder' 'accédez' 'acheter' 'achieve' 'acknowledge' 'act'
 'acting' 'action' 'actions' 'activating' 'activation' 'active' 'activer'
 'activities' 'activity' 'activité' 'activités' 'actor' 'acts' 'actualité'
 'actualités' 'actually' 'adam' 'add' 'added' 'addendum' 'adding'
 'addition' 'additional' 'address' 'addresses' 'admin' 'administration'
 'adresse' 'ads' 'adults' 'advance' 'advanced' 'adventure' 'adventures'
 'advertise' 'advertisement' 'advertisements' 'advertiser' 'advertisers'
 'advertising' 'advice' 'adweek' 'affairs' 'affect' 'affiliates'
 'afghanistan' 'afin' 'afp' 'africa' 'african' 'afrique' 'age' 'agencies'
 'agency' 'agend

In [14]:
import numpy as np

word_counts = np.asarray(X.sum(axis=0)).ravel()

words_freq = list(zip(feature_names, word_counts))

words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

print(words_freq[:20])

[('sign', np.float64(67.86750225721866)), ('cnn', np.float64(56.91100271693881)), ('share', np.float64(50.1870585448125)), ('video', np.float64(48.15887185590145)), ('password', np.float64(47.83290073998626)), ('على', np.float64(46.62786362895451)), ('les', np.float64(46.55754012760379)), ('images', np.float64(42.64427793720937)), ('guardian', np.float64(40.48432369899114)), ('des', np.float64(39.65122632461002)), ('bbc', np.float64(38.53097233689864)), ('information', np.float64(38.47641799716185)), ('whatsapp', np.float64(37.50457879195702)), ('privacy', np.float64(36.69203039047643)), ('ago', np.float64(36.3904049367062)), ('use', np.float64(36.19146168229029)), ('email', np.float64(36.17617459093715)), ('download', np.float64(36.01361206944366)), ('news', np.float64(35.36333045411299)), ('google', np.float64(34.211075712415266))]


#### Score Émotionnel — Amélioration avec VADER




In [15]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialiser l'analyseur VADER
analyzer = SentimentIntensityAnalyzer()

def emotional_score(text):
    """
    Retourne le score compound VADER entre -1.0 (négatif) et +1.0 (positif).
    Gère la négation, les majuscules, la ponctuation et les intensificateurs.
    """
    if not isinstance(text, str) or text.strip() == "":
        return 0.0

    scores = analyzer.polarity_scores(text)

    # 'compound' = score global normalisé entre -1.0 et +1.0
    return scores["compound"]

In [16]:
df["emotion_score"] = df["clean_text"].apply(emotional_score)

print(df[["clean_text", "emotion_score"]].head(10))

                                          clean_text  emotion_score
0  mondefr actualités infos france dans monde tit...        -0.6597
1  breaking news russia news world news video ico...        -0.9962
2  sputnik news world news breaking news stories ...        -0.9638
3  mots croisés par philippe dupuis monde jeux mo...         0.3818
4  globaleaks skip content error running unsuppor...         0.6124
5  cours langues ligne gymglish français english ...         0.9403
6  plus grilles jeux découvrir chaque jour une no...         0.4767
7  goût goût sport goût goût sport ttc couverture...         0.3818
8  nouvel obs actualités france dans monde récit ...        -0.7430
9  monde jeux mots croisés mini mots croisés sudo...         0.7269


In [17]:
# Bonus : scores détaillés pos / neg / neu en colonnes séparées
def emotional_scores_detail(text):
    if not isinstance(text, str) or text.strip() == "":
        return {"pos": 0.0, "neg": 0.0, "neu": 0.0, "compound": 0.0}
    return analyzer.polarity_scores(text)

df[["emotion_pos", "emotion_neg", "emotion_neu", "emotion_compound"]] = df["clean_text"].apply(
    lambda t: pd.Series(emotional_scores_detail(t))
)

print(df[["emotion_pos", "emotion_neg", "emotion_neu", "emotion_compound"]].head(10))

   emotion_pos  emotion_neg  emotion_neu  emotion_compound
0        0.017        0.973        0.010           -0.6597
1        0.159        0.781        0.060           -0.9962
2        0.168        0.704        0.128           -0.9638
3        0.000        0.983        0.017            0.3818
4        0.135        0.662        0.203            0.6124
5        0.009        0.951        0.040            0.9403
6        0.000        0.980        0.020            0.4767
7        0.000        0.981        0.019            0.3818
8        0.027        0.966        0.007           -0.7430
9        0.000        0.977        0.023            0.7269


#### Score Politique — Amélioration avec TF-IDF + Similarité Cosinus


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

# Phrase de référence décrivant le thème politique
# Enrichir cette phrase pour élargir la couverture thématique
political_reference = """
government president war nato election russia china
policy parliament democracy vote military conflict
sanctions diplomacy geopolitics international
minister congress senate legislation treaty alliance
propaganda sovereignty ideology power
"""

# Transformer la référence avec le MÊME vectorizer déjà entraîné
ref_vector = vectorizer.transform([political_reference])

# Calculer la similarité cosinus entre chaque document et la référence
similarities = cosine_similarity(X, ref_vector).ravel()

df["political_score"] = similarities

print(df[["clean_text", "political_score"]].head(10))

                                          clean_text  political_score
0  mondefr actualités infos france dans monde tit...         0.000000
1  breaking news russia news world news video ico...         0.072824
2  sputnik news world news breaking news stories ...         0.164872
3  mots croisés par philippe dupuis monde jeux mo...         0.000000
4  globaleaks skip content error running unsuppor...         0.000000
5  cours langues ligne gymglish français english ...         0.000000
6  plus grilles jeux découvrir chaque jour une no...         0.000000
7  goût goût sport goût goût sport ttc couverture...         0.000000
8  nouvel obs actualités france dans monde récit ...         0.000000
9  monde jeux mots croisés mini mots croisés sudo...         0.000000


In [19]:
# Bonus : Topic Modeling avec LDA pour découvrir les thèmes automatiquement
from sklearn.decomposition import LatentDirichletAllocation

N_TOPICS = 5

lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=10)
lda.fit(X)

# Topic dominant par document
topic_assignments = lda.transform(X).argmax(axis=1)
df["dominant_topic"] = topic_assignments

# Afficher les mots-clés de chaque topic
print("Top mots par topic :")
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"  Topic {topic_idx}: {', '.join(top_words)}")

print("\nDistribution des topics :")
print(df["dominant_topic"].value_counts())

Top mots par topic :
  Topic 0: تعريف, الارتباط, ملفات, الانتباه, المدة, يستحق, javascript, صوتي, تسجيل, youtube
  Topic 1: whatsapp, guardian, information, privacy, data, use, services, account, personal, help
  Topic 2: على, share, إلى, telegram, المدة, web, download, page, الجزيرة, فيديو
  Topic 3: les, des, pour, vous, monde, sur, une, votre, nytimescom, par
  Topic 4: cnn, sign, video, password, images, ago, news, bbc, world, cookies

Distribution des topics :
dominant_topic
4    1127
1     334
3     323
2     311
0     105
Name: count, dtype: int64


In [20]:
# Sauvegarder le DataFrame enrichi avec tous les scores améliorés
df.to_csv("datasets/features_ameliores.csv", index=False)

print("Colonnes finales :", df.columns.tolist())
print(df[["emotion_score", "emotion_pos", "emotion_neg", "political_score", "dominant_topic"]].describe())

Colonnes finales : ['url', 'http_headers', 'links', 'nb_links', 'saved_at', 'scraped_at', 'status_code', 'text', 'title', 'domain', 'label', 'clean_text', 'emotion_score', 'emotion_pos', 'emotion_neg', 'emotion_neu', 'emotion_compound', 'political_score', 'dominant_topic']
       emotion_score  emotion_pos  emotion_neg  political_score  \
count    2200.000000  2200.000000  2200.000000      2200.000000   
mean        0.397620     0.039589     0.856215         0.013772   
std         0.640139     0.071433     0.131294         0.030641   
min        -0.999100     0.000000     0.000000         0.000000   
25%         0.000000     0.000000     0.768000         0.000000   
50%         0.564150     0.008000     0.851500         0.000000   
75%         0.978100     0.056000     0.988000         0.010086   
max         0.999800     1.000000     1.000000         0.319335   

       dominant_topic  
count     2200.000000  
mean         2.924091  
std          1.298252  
min          0.000000  
25